In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Store dataset archive path
data =  "/content/drive/MyDrive/DSDDos.zip"

In [ ]:
# Install archive extraction library
pip install patool

In [ ]:
import patoolib
patoolib.extract_archive(data, outdir="/content/drive/MyDrive/DSDDos")

In [ ]:
!pip install matplotlib seaborn scikit-learn imbalanced-learn catboost lightgbm  keras shap

In [ ]:
!pip install numpy==1.23.5 pandas==1.5.3 tensorflow==2.13.0

In [ ]:
# Create requirements file
requirements = """
numpy==1.23.5
pandas==1.5.3
tensorflow==2.13.0
matplotlib
seaborn
scikit-learn
imbalanced-learn
catboost
lightgbm
keras
shap
"""

with open('/content/requirements.txt', 'w') as f:
    f.write(requirements)
    print("Requirements file created successfully.")

!cat /content/requirements.txt

In [ ]:
# Copy requirements file to Google Drive
!cp /content/requirements.txt /content/drive/MyDrive/Data/requirements.txt

In [ ]:
# Install libraries from requirements file
!pip install -r /content/drive/MyDrive/Data/requirements.txt

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# Load dataset and create a sample
import pandas as pd

file_path = "/content/drive/MyDrive/DSDDos/DSDDos.csv"

df = pd.read_csv(file_path)
df_sample = df.sample(frac= 0.7, random_state=42)

print(df_sample.head())
print(f"Shape of sampled data: {df_sample.shape}")

In [ ]:
# Save sampled dataset
df_sample.to_parquet("/content/drive/MyDrive/DSDDos/sampled_data.parquet", index=False)
print("Sampled data saved successfully as Parquet.")

In [ ]:
# Load sampled dataset
df_sample = pd.read_parquet("/content/drive/MyDrive/DSDDos/sampled_data.parquet")
print("Sampled data loaded successfully.")

In [ ]:
# Remove missing values
data_cleaned = df_sample.dropna()

In [ ]:
# Encode attack labels
data_cleaned['label'] = data_cleaned['label'].replace({
    'BenignTraffic': 0, 'DDoS-ICMP_Flood': 1, 'DDoS-UDP_Flood': 2, 'DDoS-TCP_Flood': 3, 'DDoS-PSHACK_Flood': 4,    'DDoS-SYN_Flood': 5, 'DDoS-RSTFINFlood': 6, 'DDoS-SynonymousIP_Flood': 7, 'DDoS-ICMP_Fragmentation': 8,
    'DDoS-ACK_Fragmentation': 9, 'DDoS-UDP_Fragmentation': 10, 'DDoS-HTTP_Flood': 11, 'DDoS-SlowLoris': 12,
    'DoS-UDP_Flood': 13, 'DoS-TCP_Flood': 13, 'DoS-SYN_Flood': 13, 'DoS-HTTP_Flood': 13,
    'Mirai-greeth_flood': 14, 'Mirai-udpplain': 14, 'Mirai-greip_flood': 14,
    'MITM-ArpSpoofing': 15, 'DNS_Spoofing': 15, 'Recon-HostDiscovery': 15, 'Recon-OSScan': 15, 'Recon-PortScan': 15,
    'VulnerabilityScan': 15, 'DictionaryBruteForce': 15, 'SqlInjection': 15, 'BrowserHijacking': 15,
    'CommandInjection': 15, 'Backdoor_Malware': 15, 'XSS': 15, 'Uploading_Attack': 15, 'Recon-PingSweep': 15
})

In [ ]:
# Save cleaned dataset
data_cleaned.to_parquet("/content/drive/MyDrive/Data/cleaned_data.parquet", index=False)
print("Cleaned data saved successfully as Parquet.")

In [ ]:
# Load cleaned dataset
data_cleaned = pd.read_parquet("/content/drive/MyDrive/Data/cleaned_data.parquet")
print("Cleaned data loaded successfully.")

In [ ]:
# Define class names for visualization
class_names = {
    0: 'BenignTraffic', 1:'DDoS-ICMP_Flood',2: 'DDoS-UDP_Flood', 3: 'DDoS-TCP_Flood',4: 'DDoS-PSHACK_Flood',
     5:'DDoS-SYN_Flood', 6:'DDoS-RSTFINFlood', 7:'DDoS-SynonymousIP_Flood', 8:'DDoS-ICMP_Fragmentation',
     9:'DDoS-ACK_Fragmentation', 10:'DDoS-UDP_Fragmentation', 11:'DDoS-HTTP_Flood',12: 'DDoS-SlowLoris',
    13: 'DOS Attack', 14: 'MIRAI Attack', 15: 'OTHER Attack'
}

In [ ]:
# Plot class distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=data_cleaned, x='label', order=data_cleaned['label'].value_counts().index)
plt.title('Class Distribution After Preprocessing')
# Replace class_labels with class_names
plt.xticks(ticks=range(len(class_names)), labels=[class_names[i] for i in range(len(class_names))], rotation=38)

# Save plot image
plt.savefig("/content/drive/MyDrive/Image/class_distribution_plot.png", bbox_inches='tight')
print("Class distribution plot saved successfully.")

plt.show()

In [ ]:
# Load saved plot image
img = mpimg.imread('/content/drive/MyDrive/Image/class_distribution_plot.png')

plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Prepare features and labels
X = data_cleaned.drop('label', axis=1)
y = data_cleaned['label']

In [ ]:
# Standardize features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Apply PCA for dimensionality reduction
from sklearn.decomposition import PCA

pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
# Save PCA-transformed data
np.save("/content/drive/MyDrive/Data/X_pca.npy", X_pca)
np.save("/content/drive/MyDrive/Data/y.npy", y)
print("PCA-transformed data saved successfully.")

In [ ]:
# Load PCA-transformed data
X_pca = np.load("/content/drive/MyDrive/Data/X_pca.npy")
y = np.load("/content/drive/MyDrive/Data/y.npy")
print("PCA-transformed data loaded successfully.")

In [ ]:
# Apply SMOTE for class balancing
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy='auto', random_state=42)

batch_size = 100_000
num_batches = int(np.ceil(X_pca.shape[0] / batch_size))

X_resampled_list = []
y_resampled_list = []

for i in range(num_batches):
    print(f"Processing batch {i + 1} of {num_batches}")
    start = i * batch_size
    end = min((i + 1) * batch_size, X_pca.shape[0])

    X_batch = X_pca[start:end]
    y_batch = y[start:end]

    if len(np.unique(y_batch)) > 1:
        Apply SMOTE to the current batch
        X_resampled_batch, y_resampled_batch = smote.fit_resample(X_batch, y_batch)
        X_resampled_list.append(X_resampled_batch)
        y_resampled_list.append(y_resampled_batch)
    else:
        print(f"Skipped batch {i + 1} because it contains only one class.")

X_resampled = np.vstack(X_resampled_list)
y_resampled = np.hstack(y_resampled_list)

print("Original data shape before PCA processing:", X_pca.shape)
print("Data shape after SMOTE:", X_resampled.shape)

In [ ]:
# Save preprocessed data
np.save("/content/drive/MyDrive/Data/X_resampled.npy", X_resampled)
np.save("/content/drive/MyDrive/Data/y_resampled.npy", y_resampled)
print("Final preprocessed data saved successfully.")

In [ ]:
# Load preprocessed data
X_resampled = np.load("/content/drive/MyDrive/Data/X_resampled.npy")
y_resampled = np.load("/content/drive/MyDrive/Data/y_resampled.npy")
print("Final preprocessed data loaded successfully.")

In [ ]:
# Plot class distribution after resampling
plt.figure(figsize=(10, 6))
sns.countplot(x=y_resampled, order=pd.Series(y_resampled).value_counts().index)
plt.title('Class Distribution After Resampling')
# Replace class_labels with class_names
plt.xticks(ticks=range(len(class_names)), labels=[class_names[i] for i in range(len(class_names))], rotation=45)

plt.savefig("/content/drive/MyDrive/Image/class_distribution_plot2.png", bbox_inches='tight')
print("Class distribution plot saved successfully.")

plt.show()

In [ ]:
img = mpimg.imread('/content/drive/MyDrive/Image/class_distribution_plot2.png')

plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Split data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.4, random_state=42, stratify=y_resampled)

In [ ]:
# Save training and testing datasets
np.save("/content/drive/MyDrive/Data/X_train.npy", X_train)
np.save("/content/drive/MyDrive/Data/X_test.npy", X_test)
np.save("/content/drive/MyDrive/Data/y_train.npy", y_train)
np.save("/content/drive/MyDrive/Data/y_test.npy", y_test)
print("Train-test split data saved successfully.")

In [ ]:
# Load training and testing datasets
X_train = np.load("/content/drive/MyDrive/Data/X_train.npy")
X_test = np.load("/content/drive/MyDrive/Data/X_test.npy")
y_train = np.load("/content/drive/MyDrive/Data/y_train.npy")
y_test = np.load("/content/drive/MyDrive/Data/y_test.npy")
print("Train-test split data loaded successfully.")

In [ ]:
# Train CatBoost model
from catboost import CatBoostClassifier
from joblib import dump

catboost_model = CatBoostClassifier(
    depth=10,
    iterations=1000,
    learning_rate=0.1,
    l2_leaf_reg=5,
    loss_function='MultiClass',
    random_seed=42,
    silent=False
)

catboost_model.fit(X_train, y_train)
y_pred = catboost_model.predict(X_test)

# Calculate accuracy and classification metrics
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
dump(catboost_model, '/content/drive/My Drive/Model/catboost3_model.joblib')

In [ ]:
# Display training and testing data shapes
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# Check for invalid values
print(np.isnan(X_train).sum(), np.isnan(X_test).sum())
print(np.isinf(X_train).sum(), np.isinf(X_test).sum())

In [ ]:
# Display class distribution in training data
print(np.unique(y_train, return_counts=True))

In [ ]:
# Train LSTM model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import matplotlib.pyplot as plt

# Configure Adam optimizer
custom_adam = Adam(learning_rate=0.001)

# Assume training data has been preprocessed
# Reshape X_train to be [samples, timesteps, features]
timesteps = 1
X_train_lstm = np.reshape(X_train, (X_train.shape[0], timesteps, X_train.shape[1]))
X_test_lstm = np.reshape(X_test, (X_test.shape[0], timesteps, X_test.shape[1]))

# Define LSTM architecture
lstm_model = Sequential([
    LSTM(512, return_sequences=True, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    Dropout(0.5),
    LSTM(1024, return_sequences=False),
    Dropout(0.5),
    Dense(len(np.unique(y_train)), activation='softmax')
])

# Compile the model
lstm_model.compile(optimizer=custom_adam, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Configure EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = lstm_model.fit(
    X_train_lstm, y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping]
)

# Plot training and validation accuracy
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Valid Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()


# Plot training and validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Valid Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Evaluate model on test data
y_pred_lstm = np.argmax(lstm_model.predict(X_test_lstm), axis=1)

# Display evaluation metrics
print("LSTM Model Performance:")
print("Classification Report:")
print(classification_report(y_test, y_pred_lstm))
print("Accuracy:", accuracy_score(y_test, y_pred_lstm))

# Compare actual and predicted labels
for i in range(10):  # Display first 10 predictions
    print(f"Real: {y_test[i]}, Expected: {y_pred_lstm[i]}")

In [ ]:
from joblib import dump
dump(lstm_model, '/content/drive/My Drive/Model/lstm_final_model.joblib')
print("Trained LSTM model saved successfully.")

In [ ]:
lstm_model.save('/content/drive/My Drive/Model/lstm_final_model.h5')
print("Trained LSTM model saved successfully.")

In [ ]:
# Train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train, y_train)

# Generate predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate model performance
print(" Random Forest Model Performance:")
print(classification_report(y_test, y_pred_rf))

# Plot confusion matrix
plt.figure(figsize=(10, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues')
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Calculate model accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f" Random Forest Accuracy: {accuracy_rf:.4f}")

In [ ]:
from joblib import dump
dump(rf_model, '/content/drive/My Drive/Model/random_forest_model.joblib')
print("Random Forest Model saved successfully!")

In [ ]:
# Load models and test data for stacking ensemble
print(" Loading models... ")
lstm_model = joblib.load("/content/drive/MyDrive/Model/lstm_final_model.joblib")
rf_model = joblib.load("/content/drive/MyDrive/Model/random_forest_model.joblib")
catboost_model = joblib.load("/content/drive/MyDrive/Model/catboost3_model.joblib")

print(" Loading test data... ")
X_test = np.load("/content/drive/MyDrive/Data/X_test.npy")
y_test = np.load("/content/drive/MyDrive/Data/y_test.npy")

In [ ]:
# Train LightGBM stacking ensemble model
from lightgbm import LGBMClassifier
from tqdm import tqdm

print(" Generating probability predictions from base models...")
# Reshape X_test to include the timesteps dimension
X_test_reshaped = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
y_pred_lstm = lstm_model.predict(X_test_reshaped)  # Use the reshaped data
y_pred_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_catboost = catboost_model.predict_proba(X_test)[:, 1]

print(" Stacking predictions...")
stacked_features = np.column_stack((y_pred_lstm, y_pred_rf, y_pred_catboost))

print(" Training LGBM Meta-Model...")
meta_model = LGBMClassifier(
    n_estimators=500,  # Increase number of trees to improve performance
    max_depth=7,       # Increase depth to capture complex patterns
    learning_rate=0.05, # Lower learning rate for stable training
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    lambda_l1=0.1,
    lambda_l2=0.1,
    verbose=1
)

Use tqdm to monitor training progress
for _ in tqdm(range(1), desc="Training Progress"):
    meta_model.fit(stacked_features, y_test)

print(" Making final predictions...")
y_pred_final = meta_model.predict(stacked_features)

# Calculate final ensemble accuracy
accuracy = accuracy_score(y_test, y_pred_final)
print(f" Final Stacking Model Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# Save stacking ensemble model
joblib.dump(meta_model, "/content/drive/MyDrive/Model/meta_model_lgbm1.joblib")

In [ ]:
# Download trained models
from google.colab import files
files.download('catboost3_model.joblib')
files.download('lstm_final_model.joblib')
files.download('random_forest_model.joblib')
files.download('meta_model_lgbm1.joblib')